In [2]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================================================================
# 0. DIRECTORY PATH FIX 
# =========================================================================
# This appends the parent folder (the root of the repo) to Python's path
# so it can successfully find and import the 'models' directory from inside 
# your 'research' folder.
sys.path.append(os.path.abspath('..')) 

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp
from models.frameworks import IsoAlign

In [8]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [9]:
sys.path.append(os.path.abspath('..')) 

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp
from models.frameworks import IsoAlign

In [6]:
print("working4")

working4


In [10]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder
from models.frameworks import IsoAlign

# =========================================================================
# NEW: SEQUENTIAL FUSION BiLSTM FOR WAVELETS (Foolproof Shape Router)
# =========================================================================
class SequentialFusionBiLSTM(nn.Module):
    def __init__(self, in_channels, scales, hidden_dim, out_dim, time_steps=48):
        super().__init__()
        self.in_channels = in_channels
        self.scales = scales
        self.time_steps = time_steps # Explicitly track the 48 time steps
        
        # Flattened dimension per time step: C x Scales (3 * 64 = 192)
        self.input_dim = in_channels * scales 
        
        # Sequence Modeling: BiLSTM to capture forward/backward dynamics
        self.bilstm = nn.LSTM(
            input_size=self.input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # Project the BiLSTM output to match the IsoAlign latent dimension
        self.fc = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, x):
        # The parent IsoAlign framework is shuffling the dimensions upstream.
        # We will dynamically hunt down where it put our C(3), Scales(64), and L(48)
        # using their unique sizes.
        dims = list(x.shape)
        B = dims[0]
        
        # Find the axes by looking for their size (starting from index 1 to skip Batch)
        c_idx = dims.index(self.in_channels, 1)
        s_idx = dims.index(self.scales, 1)
        l_idx = dims.index(self.time_steps, 1)
        
        # Force the tensor into EXACTLY: (Batch, Time, Channels, Scales)
        # This guarantees it becomes (B, 48, 3, 64) regardless of upstream transposes
        x = x.permute(0, l_idx, c_idx, s_idx).contiguous() 
        
        # Flatten the spatial/channel dimensions safely
        # New shape: (B, 48, 192)
        x = x.view(B, self.time_steps, self.input_dim) 
        
        # Sequence Modeling
        out, _ = self.bilstm(x) # out shape: (B, L, hidden_dim * 2)
        
        # Pool across the temporal dimension (L) to get a single vector per batch item
        out_pooled = torch.max(out, dim=1)[0]
        
        # Map to LATENT_DIM
        final_rep = self.fc(out_pooled)
        
        return None, final_rep
# =========================================================================
# 1. FIXED DATASET CLASS
# =========================================================================
class SleepEDF_HF_Dataset(Dataset):
    def __init__(self, pt_file_path="/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt", split="train"):
        print(f"Loading Hugging Face dataset from {pt_file_path}...")
        
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                data_obj = data_obj[split]
                
            if "samples" in data_obj:
                self.data = data_obj["samples"]
            elif "data" in data_obj:
                self.data = data_obj["data"]
            elif "x_data" in data_obj:
                self.data = data_obj["x_data"]
            elif "X_train" in data_obj:
                self.data = data_obj["X_train"]
            else:
                raise ValueError(f"Could not find the tensor. Available keys: {data_obj.keys()}")
        else:
            self.data = data_obj
            
        if not isinstance(self.data, torch.Tensor):
            self.data = torch.FloatTensor(self.data)
        else:
            self.data = self.data.float()
            
        if self.data.dim() == 2:
            self.data = self.data.unsqueeze(1)
            
        print(f"✅ Successfully loaded {len(self.data)} epochs from split '{split}'.")
        self.window = torch.hann_window(128)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        x_t = self.data[idx] 
        x_time = x_t 
        
        x_fft = torch.fft.rfft(x_t, dim=-1)
        magnitude = torch.abs(x_fft)
        phase = torch.angle(x_fft)
        x_fourier = torch.cat([magnitude, phase], dim=0)
        
        x_stft = torch.stft(x_t, n_fft=128, hop_length=64, window=self.window, return_complex=True)
        x_wavelet = torch.abs(x_stft)[:, :64, :] 
        x_wavelet = F.pad(x_wavelet, (0, 1)) 
        
        return x_time, x_fourier, x_wavelet

# =========================================================================
# 2. SETUP HYPERPARAMETERS & DEVICE CONFIGURATION
# =========================================================================
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu") 
print(f"\nUsing device: {DEVICE}")

N_CHANNELS = 3 
TIME_CHANNELS = N_CHANNELS
FT_CHANNELS = N_CHANNELS * 2 

BATCH_SIZE = 64
EPOCHS = 40
TIME_STEPS = 3000    
LATENT_DIM = 128     

class Args:
    wo_OB = False
    wo_OF = False
args = Args()

# =========================================================================
# 3. INITIALIZE DATA LOADERS
# =========================================================================
dataset_path = "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt"
train_dataset = SleepEDF_HF_Dataset(pt_file_path=dataset_path, split="train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

_, _, sample_w = train_dataset[0]
SPECT_FREQ = sample_w.shape[1]   
SPECT_TIME = sample_w.shape[2]   

print(f"Spectrogram dimensions configured to: {SPECT_FREQ} Freqs x {SPECT_TIME} Time steps")

# =========================================================================
# 4. INSTANTIATE ARCHITECTURE COMPONENTS
# =========================================================================
# A. Time Encoder (1D ResNet)
time_encoder = ResNet1D(
    in_channels=TIME_CHANNELS, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, 
    use_do=True, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

# B. Wavelet Encoder (NEW: Sequential Fusion BiLSTM)
# Replaces the previous UNET_2D_simp implementation
spect_encoder = SequentialFusionBiLSTM(
    in_channels=N_CHANNELS, 
    scales=SPECT_FREQ, 
    hidden_dim=64, 
    out_dim=LATENT_DIM
).to(DEVICE)

# C. Fourier Encoder Wrapper
class FourierWrapper(nn.Module):
    def __init__(self, in_length):
        super().__init__()
        self.enc = FourierEncoder(in_channels=FT_CHANNELS, in_length=in_length, out_channels=LATENT_DIM)
        if in_length == 3000:
            self.enc.fc_abs = nn.Linear(376, 1)
            self.enc.fc_angle = nn.Linear(376, 1)
            
    def forward(self, x):
        return None, self.enc(x)

ft_encoder = FourierWrapper(in_length=TIME_STEPS).to(DEVICE)

# D. Master IsoAlign Multi-View Framework
model = IsoAlign(
    backbone=time_encoder, spect_encoder=spect_encoder, FT_encoder=ft_encoder, 
    DEVICE=DEVICE, dim=LATENT_DIM, batch_size=BATCH_SIZE, args=args
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)




Using device: cuda:1
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 265575 epochs from split 'train'.
Spectrogram dimensions configured to: 64 Freqs x 48 Time steps


In [4]:
# =========================================================================
# 5. EXECUTE THE TRAINING LOOP
# =========================================================================
print("\n🚀 Starting Self-Supervised Sequential Fusion Pre-training on GPU-1...")

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_loss = float('inf')
save_every_n_epochs = 5 

# --- CHECKPOINT RECOVERY ---
start_epoch = 0
resume_path = os.path.join(checkpoint_dir, "best_sequential_bilstm.pth") 

if os.path.exists(resume_path):
    print(f"\n🔄 Found checkpoint at {resume_path}. Loading...")
    checkpoint = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(checkpoint) 
    print("✅ Model weights loaded successfully.")
else:
    print("\n⚠️ No checkpoint found. Starting training from scratch.")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w) in enumerate(train_loader):
        batch_t = batch_t.to(DEVICE)
        batch_f = batch_f.to(DEVICE)
        
        # The Wavelet tensor is passed in its native shape: (B, C, Scales, L)
        # The SequentialFusionBiLSTM now securely handles all reshaping natively.
        batch_w = batch_w.to(DEVICE) 
        
        optimizer.zero_grad()
        loss = model(batch_t, batch_w, batch_f)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Average Loss: {avg_loss:.4f}")
    
    # Checkpoint Metric Tracking
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_model_path = os.path.join(checkpoint_dir, "best_sequential_bilstm.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"   🌟 New best loss achieved! Saved encoder to: {best_model_path}")
        
    if (epoch + 1) % save_every_n_epochs == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"sequential_bilstm_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_path)
        print(f"   (i) Full recovery checkpoint saved to: {checkpoint_path}")

print("\n🎉 Pre-training Complete!")


🚀 Starting Self-Supervised Sequential Fusion Pre-training on GPU-1...

🔄 Found checkpoint at checkpoints/best_sequential_bilstm.pth. Loading...
✅ Model weights loaded successfully.


KeyboardInterrupt: 

In [5]:
print("\n🚀 Starting Self-Supervised Sequential Fusion Pre-training on GPU-1...")

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_loss = float('inf')
save_every_n_epochs = 5 
log_interval = 50

start_epoch = 0
resume_path = os.path.join(checkpoint_dir, "best_sequential_bilstm.pth") 

if os.path.exists(resume_path):
    print(f"\n🔄 Found checkpoint at {resume_path}. Loading...")
    checkpoint = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(checkpoint) 
    print("✅ Model weights loaded successfully.")
else:
    print("\n⚠️ No checkpoint found. Starting training from scratch.")

if hasattr(model, 'bilstm'):
    model.bilstm.flatten_parameters()

model = torch.compile(model)
scaler = torch.amp.GradScaler("cuda")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w) in enumerate(train_loader):
        batch_t = batch_t.to(DEVICE, non_blocking=True)
        batch_f = batch_f.to(DEVICE, non_blocking=True)
        batch_w = batch_w.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast("cuda"):
            loss = model(batch_t, batch_w, batch_f)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % log_interval == 0:
            current_batch_loss = loss.item()
            running_avg_loss = total_loss / (batch_idx + 1)
            
            print(f"  Epoch: {epoch+1} [{batch_idx + 1}/{len(train_loader)}] | Batch Loss: {current_batch_loss:.4f} | Running Avg: {running_avg_loss:.4f}")
            
    avg_loss = total_loss / len(train_loader)
    print(f"=== Epoch [{epoch+1}/{EPOCHS}] Completed | Final Average Loss: {avg_loss:.4f} ===\n")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_model_path = os.path.join(checkpoint_dir, "best_sequential_bilstm.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"   🌟 New best loss achieved! Saved encoder to: {best_model_path}")
        
    if (epoch + 1) % save_every_n_epochs == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"sequential_bilstm_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_path)
        print(f"   (i) Full recovery checkpoint saved to: {checkpoint_path}")

print("\n🎉 Pre-training Complete!")


🚀 Starting Self-Supervised Sequential Fusion Pre-training on GPU-1...

🔄 Found checkpoint at checkpoints/best_sequential_bilstm.pth. Loading...
✅ Model weights loaded successfully.


W0706 23:16:18.926000 1590862 torch/_inductor/utils.py:1137] [0/0_1] Not enough SMs to use max_autotune_gemm mode
/home/gella.saikrishna/.venv/lib/python3.12/site-packages/torch/_inductor/lowering.py:1625: FutureWarning: `torch._prims_common.check` is deprecated and will be removed in the future. Please use `torch._check*` functions instead.
  check(
/home/gella.saikrishna/.venv/lib/python3.12/site-packages/torch/_inductor/lowering.py:1625: FutureWarning: `torch._prims_common.check` is deprecated and will be removed in the future. Please use `torch._check*` functions instead.
  check(


  Epoch: 1 [50/4149] | Batch Loss: 5.4693 | Running Avg: 4.4646
  Epoch: 1 [100/4149] | Batch Loss: 3.8101 | Running Avg: 4.3671
  Epoch: 1 [150/4149] | Batch Loss: 4.7017 | Running Avg: 4.2631
  Epoch: 1 [200/4149] | Batch Loss: 3.5358 | Running Avg: 4.1463
  Epoch: 1 [250/4149] | Batch Loss: 5.5422 | Running Avg: 4.1411
  Epoch: 1 [300/4149] | Batch Loss: 4.4414 | Running Avg: 4.1339
  Epoch: 1 [350/4149] | Batch Loss: 4.7813 | Running Avg: 4.1642
  Epoch: 1 [400/4149] | Batch Loss: 4.3980 | Running Avg: 4.1686
  Epoch: 1 [450/4149] | Batch Loss: 4.5102 | Running Avg: 4.1516
  Epoch: 1 [500/4149] | Batch Loss: 3.9464 | Running Avg: 4.1567
  Epoch: 1 [550/4149] | Batch Loss: 3.7684 | Running Avg: 4.1428
  Epoch: 1 [600/4149] | Batch Loss: 3.5853 | Running Avg: 4.1171
  Epoch: 1 [650/4149] | Batch Loss: 4.0844 | Running Avg: 4.1238
  Epoch: 1 [700/4149] | Batch Loss: 4.1382 | Running Avg: 4.1172
  Epoch: 1 [750/4149] | Batch Loss: 3.7208 | Running Avg: 4.1068
  Epoch: 1 [800/4149] | Ba

KeyboardInterrupt: 

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix

class SleepEDF_Downstream_Dataset(SleepEDF_HF_Dataset):
    def __init__(self, pt_file_path, split="train"):
        super().__init__(pt_file_path=pt_file_path, split=split)
        
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                data_obj = data_obj[split]
            
            if "labels" in data_obj:
                self.labels = data_obj["labels"]
            elif "y_data" in data_obj:
                self.labels = data_obj["y_data"]
            else:
                raise ValueError(f"Could not find labels in {data_obj.keys()}")
        else:
            raise ValueError("Dataset file must be a dictionary to contain both data and labels.")
            
        if not isinstance(self.labels, torch.Tensor):
            self.labels = torch.tensor(self.labels, dtype=torch.long)
            
        print(f"Labels loaded for split '{split}'. Shape: {self.labels.shape}")

    def __getitem__(self, idx):
        x_time, x_fourier, x_wavelet = super().__getitem__(idx)
        label = self.labels[idx]
        return x_time, x_fourier, x_wavelet, label

torch.manual_seed(42)
np.random.seed(42)

full_train_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="train")
subset_size = int(0.10 * len(full_train_dataset))
indices = np.arange(len(full_train_dataset))
np.random.shuffle(indices)
train_subset = Subset(full_train_dataset, indices[:subset_size])
train_loader_10 = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)

print("Loading Test Dataset...")
test_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="test")
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False) 

state_dict = torch.load("checkpoints/best_sequential_bilstm.pth", map_location=DEVICE)

new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("_orig_mod."):
        name = k[len("_orig_mod."):]
    else:
        name = k
    new_state_dict[name] = v

model.load_state_dict(new_state_dict)

for param in model.parameters():
    param.requires_grad = False

class MultiViewClassifier(nn.Module):
    def __init__(self, encoder_model, latent_dim, num_classes):
        super().__init__()
        self.encoder = encoder_model
        self.classifier = nn.Sequential(
            nn.Linear(3 * latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x1, x2, x3):
        _, R_t = self.encoder.encoder(x1)
        _, R_f = self.encoder.spect_encoder(x2)
        _, R_f_FT = self.encoder.FT_encoder(x3)
        
        R_t = self.encoder.projector(R_t)
        R_f = self.encoder.projector_spect(R_f)
        R_f_FT = self.encoder.projector_FT(R_f_FT)
        
        combined = torch.cat([R_t, R_f, R_f_FT], dim=1)
        return self.classifier(combined)

clf_model = MultiViewClassifier(model, LATENT_DIM, num_classes=5).to(DEVICE)
optimizer = optim.Adam(clf_model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Starting downstream fine-tuning with 10% data and test evaluation...")
log_interval = 50

for epoch in range(20):
    clf_model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w, labels) in enumerate(train_loader_10):
        batch_t = batch_t.to(DEVICE).transpose(1, 2)
        batch_w = batch_w.to(DEVICE) 
        batch_f = batch_f.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = clf_model(batch_t, batch_w, batch_f)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % log_interval == 0:
            current_avg_loss = total_loss / (batch_idx + 1)
            print(f"  Train Epoch: {epoch+1} [{batch_idx * len(batch_t)}/{len(train_loader_10.dataset)} "
                  f"({100. * batch_idx / len(train_loader_10):.0f}%)]\tLoss: {current_avg_loss:.4f}")
            
    avg_loss = total_loss / len(train_loader_10)
    
    clf_model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_t, batch_f, batch_w, labels in test_loader:
            batch_t = batch_t.to(DEVICE).transpose(1, 2)
            batch_w = batch_w.to(DEVICE) 
            batch_f = batch_f.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = clf_model(batch_t, batch_w, batch_f)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    test_accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/20] | Train Loss: {avg_loss:.4f} | Test Acc: {test_accuracy:.2f}%\n")

print("\n" + "="*50)
print("RUNNING FINAL DETAILED TEST EVALUATION")
print("="*50)

clf_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_t, batch_f, batch_w, labels in test_loader:
        batch_t = batch_t.to(DEVICE).transpose(1, 2)
        batch_w = batch_w.to(DEVICE) 
        batch_f = batch_f.to(DEVICE)
        
        outputs = clf_model(batch_t, batch_w, batch_f)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

target_names = ['Wake (0)', 'N1 (1)', 'N2 (2)', 'N3 (3)', 'REM (4)']

print("\nPER-CLASS METRICS (TEST SET)")
report = classification_report(all_targets, all_preds, target_names=target_names, digits=4)
print(report)

print("\nCONFUSION MATRIX")
conf_matrix = confusion_matrix(all_targets, all_preds)
print(conf_matrix)

Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 265575 epochs from split 'train'.
Labels loaded for split 'train'. Shape: torch.Size([265575])
Loading Test Dataset...
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 82993 epochs from split 'test'.
Labels loaded for split 'test'. Shape: torch.Size([82993])
Starting downstream fine-tuning with 10% data and test evaluation...
  Train Epoch: 1 [3136/26557 (12%)]	Loss: 1.1421
  Train Epoch: 1 [6336/26557 (24%)]	Loss: 0.9262
  Train Epoch: 1 [9536/26557 (36%)]	Loss: 0.8184
  Train Epoch: 1 [12736/26557 (48%)]	Loss: 0.7523
  Train Epoch: 1 [15936/26557 (60%)]	Loss: 0.7112
  Train Epoch: 1 [19136/26557 (72%)]	Loss: 0.6788
  Train Epoch: 1 [22336/26557 (84%)]	Loss: 0.6586
  Train Epoch: 1 [25536/26557 (96%)]	Loss: 0.6407


KeyboardInterrupt: 

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix

class SleepEDF_Downstream_Dataset(SleepEDF_HF_Dataset):
    def __init__(self, pt_file_path, split="train"):
        super().__init__(pt_file_path=pt_file_path, split=split)
        
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                data_obj = data_obj[split]
            
            if "labels" in data_obj:
                self.labels = data_obj["labels"]
            elif "y_data" in data_obj:
                self.labels = data_obj["y_data"]
            else:
                raise ValueError(f"Could not find labels in {data_obj.keys()}")
        else:
            raise ValueError("Dataset file must be a dictionary to contain both data and labels.")
            
        if not isinstance(self.labels, torch.Tensor):
            self.labels = torch.tensor(self.labels, dtype=torch.long)
            
        print(f"Labels loaded for split '{split}'. Shape: {self.labels.shape}")

    def __getitem__(self, idx):
        x_time, x_fourier, x_wavelet = super().__getitem__(idx)
        label = self.labels[idx]
        return x_time, x_fourier, x_wavelet, label

torch.manual_seed(42)
np.random.seed(42)

full_train_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="train")
subset_size = int(0.10 * len(full_train_dataset))
indices = np.arange(len(full_train_dataset))
np.random.shuffle(indices)
train_subset = Subset(full_train_dataset, indices[:subset_size])

train_loader_10 = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

print("Loading Test Dataset...")
test_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="test")

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True) 

state_dict = torch.load("checkpoints/best_sequential_bilstm.pth", map_location=DEVICE)

new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("_orig_mod."):
        name = k[len("_orig_mod."):]
    else:
        name = k
    new_state_dict[name] = v

model.load_state_dict(new_state_dict)

for param in model.parameters():
    param.requires_grad = False

class MultiViewClassifier(nn.Module):
    def __init__(self, encoder_model, latent_dim, num_classes):
        super().__init__()
        self.encoder = encoder_model
        self.classifier = nn.Sequential(
            nn.Linear(3 * latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x1, x2, x3):
        with torch.no_grad():
            _, R_t = self.encoder.encoder(x1)
            _, R_f = self.encoder.spect_encoder(x2)
            _, R_f_FT = self.encoder.FT_encoder(x3)
            
            R_t = self.encoder.projector(R_t)
            R_f = self.encoder.projector_spect(R_f)
            R_f_FT = self.encoder.projector_FT(R_f_FT)
            
            combined = torch.cat([R_t, R_f, R_f_FT], dim=1)
            
        return self.classifier(combined)

clf_model = MultiViewClassifier(model, LATENT_DIM, num_classes=5).to(DEVICE)
optimizer = optim.Adam(clf_model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

print("Starting downstream fine-tuning with 10% data and test evaluation...")
log_interval = 50

for epoch in range(20):
    clf_model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w, labels) in enumerate(train_loader_10):
        batch_t = batch_t.to(DEVICE, non_blocking=True).transpose(1, 2)
        batch_w = batch_w.to(DEVICE, non_blocking=True) 
        batch_f = batch_f.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda'):
            outputs = clf_model(batch_t, batch_w, batch_f)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % log_interval == 0:
            current_avg_loss = total_loss / (batch_idx + 1)
            print(f"  Train Epoch: {epoch+1} [{batch_idx * len(batch_t)}/{len(train_loader_10.dataset)} "
                  f"({100. * batch_idx / len(train_loader_10):.0f}%)]\tLoss: {current_avg_loss:.4f}")
            
    avg_loss = total_loss / len(train_loader_10)
    
    clf_model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_t, batch_f, batch_w, labels in test_loader:
            batch_t = batch_t.to(DEVICE, non_blocking=True).transpose(1, 2)
            batch_w = batch_w.to(DEVICE, non_blocking=True) 
            batch_f = batch_f.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = clf_model(batch_t, batch_w, batch_f)
                
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    test_accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/20] | Train Loss: {avg_loss:.4f} | Test Acc: {test_accuracy:.2f}%\n")

print("\n" + "="*50)
print("RUNNING FINAL DETAILED TEST EVALUATION")
print("="*50)

clf_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_t, batch_f, batch_w, labels in test_loader:
        batch_t = batch_t.to(DEVICE, non_blocking=True).transpose(1, 2)
        batch_w = batch_w.to(DEVICE, non_blocking=True) 
        batch_f = batch_f.to(DEVICE, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            outputs = clf_model(batch_t, batch_w, batch_f)
            
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

target_names = ['Wake (0)', 'N1 (1)', 'N2 (2)', 'N3 (3)', 'REM (4)']

print("\nPER-CLASS METRICS (TEST SET)")
report = classification_report(all_targets, all_preds, target_names=target_names, digits=4)
print(report)

print("\nCONFUSION MATRIX")
conf_matrix = confusion_matrix(all_targets, all_preds)
print(conf_matrix)

Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 265575 epochs from split 'train'.
Labels loaded for split 'train'. Shape: torch.Size([265575])
Loading Test Dataset...
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 82993 epochs from split 'test'.
Labels loaded for split 'test'. Shape: torch.Size([82993])
Starting downstream fine-tuning with 10% data and test evaluation...
  Train Epoch: 1 [3136/26557 (12%)]	Loss: 1.1466
  Train Epoch: 1 [6336/26557 (24%)]	Loss: 0.9331
  Train Epoch: 1 [9536/26557 (36%)]	Loss: 0.8260
  Train Epoch: 1 [12736/26557 (48%)]	Loss: 0.7604
  Train Epoch: 1 [15936/26557 (60%)]	Loss: 0.7198
  Train Epoch: 1 [19136/26557 (72%)]	Loss: 0.6880
  Train Epoch: 1 [22336/26557 (84%)]	Loss: 0.6679
  Train Epoch: 1 [25536/26557 (96%)]	Loss: 0.6508
Epoch [1/2

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix

class SleepEDF_Downstream_Dataset(SleepEDF_HF_Dataset):
    def __init__(self, pt_file_path, split="train"):
        super().__init__(pt_file_path=pt_file_path, split=split)
        
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                data_obj = data_obj[split]
            
            if "labels" in data_obj:
                self.labels = data_obj["labels"]
            elif "y_data" in data_obj:
                self.labels = data_obj["y_data"]
            else:
                raise ValueError(f"Could not find labels in {data_obj.keys()}")
        else:
            raise ValueError("Dataset file must be a dictionary to contain both data and labels.")
            
        if not isinstance(self.labels, torch.Tensor):
            self.labels = torch.tensor(self.labels, dtype=torch.long)
            
        print(f"Labels loaded for split '{split}'. Shape: {self.labels.shape}")

    def __getitem__(self, idx):
        x_time, x_fourier, x_wavelet = super().__getitem__(idx)
        label = self.labels[idx]
        return x_time, x_fourier, x_wavelet, label

torch.manual_seed(42)
np.random.seed(42)

full_train_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="train")
subset_size = int(0.10 * len(full_train_dataset))
indices = np.arange(len(full_train_dataset))
np.random.shuffle(indices)
train_subset = Subset(full_train_dataset, indices[:subset_size])

train_loader_10 = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

print("Loading Test Dataset...")
test_dataset = SleepEDF_Downstream_Dataset(pt_file_path=dataset_path, split="test")

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True) 

state_dict = torch.load("checkpoints/best_sequential_bilstm.pth", map_location=DEVICE)

new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("_orig_mod."):
        name = k[len("_orig_mod."):]
    else:
        name = k
    new_state_dict[name] = v

model.load_state_dict(new_state_dict)

for param in model.parameters():
    param.requires_grad = False

class MultiViewClassifier(nn.Module):
    def __init__(self, encoder_model, latent_dim, num_classes):
        super().__init__()
        self.encoder = encoder_model
        self.classifier = nn.Sequential(
            nn.Linear(3 * latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x1, x2, x3):
        with torch.no_grad():
            _, R_t = self.encoder.encoder(x1)
            _, R_f = self.encoder.spect_encoder(x2)
            _, R_f_FT = self.encoder.FT_encoder(x3)
            
            R_t = self.encoder.projector(R_t)
            R_f = self.encoder.projector_spect(R_f)
            R_f_FT = self.encoder.projector_FT(R_f_FT)
            
            combined = torch.cat([R_t, R_f, R_f_FT], dim=1)
            
        return self.classifier(combined)

clf_model = MultiViewClassifier(model, LATENT_DIM, num_classes=5).to(DEVICE)
optimizer = optim.Adam(clf_model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

print("Starting downstream fine-tuning with 10% data and test evaluation...")
log_interval = 50

for epoch in range(20):
    clf_model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w, labels) in enumerate(train_loader_10):
        batch_t = batch_t.to(DEVICE, non_blocking=True).transpose(1, 2)
        batch_w = batch_w.to(DEVICE, non_blocking=True) 
        batch_f = batch_f.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda'):
            outputs = clf_model(batch_t, batch_w, batch_f)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % log_interval == 0:
            current_avg_loss = total_loss / (batch_idx + 1)
            print(f"  Train Epoch: {epoch+1} [{batch_idx * len(batch_t)}/{len(train_loader_10.dataset)} "
                  f"({100. * batch_idx / len(train_loader_10):.0f}%)]\tLoss: {current_avg_loss:.4f}")
            
    avg_loss = total_loss / len(train_loader_10)
    
    clf_model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_t, batch_f, batch_w, labels in test_loader:
            batch_t = batch_t.to(DEVICE, non_blocking=True).transpose(1, 2)
            batch_w = batch_w.to(DEVICE, non_blocking=True) 
            batch_f = batch_f.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = clf_model(batch_t, batch_w, batch_f)
                
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    test_accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/20] | Train Loss: {avg_loss:.4f} | Test Acc: {test_accuracy:.2f}%\n")

print("\n" + "="*50)
print("RUNNING FINAL DETAILED TEST EVALUATION")
print("="*50)

clf_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_t, batch_f, batch_w, labels in test_loader:
        batch_t = batch_t.to(DEVICE, non_blocking=True).transpose(1, 2)
        batch_w = batch_w.to(DEVICE, non_blocking=True) 
        batch_f = batch_f.to(DEVICE, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            outputs = clf_model(batch_t, batch_w, batch_f)
            
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

target_names = ['Wake (0)', 'N1 (1)', 'N2 (2)', 'N3 (3)', 'REM (4)']

print("\nPER-CLASS METRICS (TEST SET)")
report = classification_report(all_targets, all_preds, target_names=target_names, digits=4)
print(report)

print("\nCONFUSION MATRIX")
conf_matrix = confusion_matrix(all_targets, all_preds)
print(conf_matrix)

Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 265575 epochs from split 'train'.
Labels loaded for split 'train'. Shape: torch.Size([265575])
Loading Test Dataset...
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 82993 epochs from split 'test'.
Labels loaded for split 'test'. Shape: torch.Size([82993])
Starting downstream fine-tuning with 10% data and test evaluation...
  Train Epoch: 1 [3136/26557 (12%)]	Loss: 1.1633
  Train Epoch: 1 [6336/26557 (24%)]	Loss: 0.9427
  Train Epoch: 1 [9536/26557 (36%)]	Loss: 0.8323
  Train Epoch: 1 [12736/26557 (48%)]	Loss: 0.7655
  Train Epoch: 1 [15936/26557 (60%)]	Loss: 0.7225
  Train Epoch: 1 [19136/26557 (72%)]	Loss: 0.6904
  Train Epoch: 1 [22336/26557 (84%)]	Loss: 0.6702
  Train Epoch: 1 [25536/26557 (96%)]	Loss: 0.6526
Epoch [1/2

In [2]:
print("Jai, Hanuman")

Jai, Hanuman
